In [1]:

import yaml
import torch
from pathlib import Path
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

from qcm.model.reupload import HybridReuploadClassifier, ReuploadClassifier
from qcm.model.reupload import QuantumHeadReupload
from qcm.data.datasets import get_dataloaders

from train_reupload import train_epoch, validate

In [3]:
def load_config(path: str) -> dict:
    with open(path, 'r') as f:
        return yaml.safe_load(f)
config = load_config('../configs/config_reupload.yaml')
dataset_type = config.get('dataset_type', 'pcam')

In [4]:
train_loader, val_loader = get_dataloaders(config)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ReuploadClassifier(config=config, use_quantum=True)
model.to(device)

ReuploadClassifier(
  (head): QuantumHeadReupload(
    (pad_layer): ZeroPad1d((0, 0))
  )
)

In [6]:
# Setup optimizer, scheduler, and loss function
optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=config['training']['lr'])
scheduler = CosineAnnealingLR(optimizer, T_max=config['training']['epochs'])

In [7]:
# Define run name and log file path
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"{dataset_type}_quantum_{timestamp}"
log_filepath = Path("./logs/training_log.csv")
log_filepath.parent.mkdir(exist_ok=True)

In [10]:
writer = SummaryWriter(log_dir=f"runs/test")
train_epoch_losses = []
val_epoch_losses = []

In [11]:
for epoch in range(config['training']['epochs']):
    avg_train_loss = train_epoch(model, train_loader, optimizer, scheduler, epoch, config, writer)
    avg_val_loss = validate(model, val_loader, epoch, config, writer)
    
    train_epoch_losses.append(avg_train_loss)
    val_epoch_losses.append(avg_val_loss)

2025-10-06 16:01:56,593 - INFO - Starting training epoch 0
2025-10-06 16:01:56,593 - INFO - Epoch 0 START
2025-10-06 16:01:57,177 - INFO - Epoch 0 END: Loss=0.2357, F1 Score=0.5171
2025-10-06 16:01:57,178 - INFO - Finished training epoch 0. Avg Loss: 0.3982, Accuracy: 0.5171
2025-10-06 16:01:57,178 - INFO - Starting validation for epoch 0
2025-10-06 16:01:57,205 - INFO - Validation Epoch 0: Loss=0.2303, F1 Score=0.1351
2025-10-06 16:01:57,206 - INFO - Starting training epoch 1
2025-10-06 16:01:57,206 - INFO - Epoch 1 START
2025-10-06 16:01:57,716 - INFO - Epoch 1 END: Loss=0.2227, F1 Score=0.8231
2025-10-06 16:01:57,716 - INFO - Finished training epoch 1. Avg Loss: 0.1980, Accuracy: 0.8231
2025-10-06 16:01:57,717 - INFO - Starting validation for epoch 1
2025-10-06 16:01:57,743 - INFO - Validation Epoch 1: Loss=0.3740, F1 Score=0.8189
2025-10-06 16:01:57,744 - INFO - Starting training epoch 2
2025-10-06 16:01:57,744 - INFO - Epoch 2 START
2025-10-06 16:01:58,251 - INFO - Epoch 2 END: Lo

In [13]:
import pennylane as qml
inp = train_loader.dataset[0][0]
label = train_loader.dataset[0][1]
dm_label = model.head.target_density_matrices[label.int()].squeeze()
params = model.head.q_params
print(qml.draw(model.head.circuit)(inp, params, dm_label))

0: ──Rot(4.78,4.82,4.84)──Rot(-0.19,-2.24,-0.01)─╭●────╭X──Rot(4.78,4.82,4.84) ···
1: ──Rot(4.83,4.80,4.83)──Rot(1.49,0.64,1.71)────╰X─╭●─│───Rot(4.83,4.80,4.83) ···
2: ──Rot(4.81,4.81,4.83)──Rot(-1.74,-3.47,-0.00)────╰X─╰●──Rot(4.81,4.81,4.83) ···

0: ··· ──Rot(1.33,0.11,0.19)───╭●────╭X──Rot(4.78,4.82,4.84)──Rot(-1.68,0.93,1.25)──╭●────╭X─┤ ···
1: ··· ──Rot(1.29,-0.86,0.13)──╰X─╭●─│───Rot(4.83,4.80,4.83)──Rot(-1.57,1.43,0.17)──╰X─╭●─│──┤ ···
2: ··· ──Rot(0.06,-0.96,-1.45)────╰X─╰●──Rot(4.81,4.81,4.83)──Rot(-0.00,1.65,-0.55)────╰X─╰●─┤ ···

0: ···  ╭<𝓗(M0)>
1: ···  ├<𝓗(M0)>
2: ···  ╰<𝓗(M0)>

M0 = 
tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


In [72]:
x = torch.tensor(train_loader.dataset[:][0]).repeat_interleave(2, 0)

In [73]:
train_loader.dataset[:][0].shape

(1024, 9)

In [74]:
x.shape

torch.Size([2048, 9])

In [75]:
label = train_loader.dataset[:][1]

In [76]:
y = model.head.target_labels.reshape(1, -1)
y = torch.repeat_interleave(y, 1024, dim=0).reshape(-1, 1)

In [77]:
y.shape

torch.Size([2048, 1])

In [78]:
_, pred = model(x,y).reshape(-1,2).min(dim=1)

In [79]:
from torchmetrics.classification import BinaryF1Score
metric = BinaryF1Score()
metric.update(pred, label)
metric_val = metric.compute()
metric_val.item()

0.8389923572540283

In [80]:
pred, label

(tensor([0, 1, 1,  ..., 0, 0, 0]), tensor([0., 1., 1.,  ..., 1., 0., 0.]))

In [81]:
xval = torch.tensor(val_loader.dataset[:10][0]).repeat_interleave(2, 0)

In [82]:
val_loader.dataset[:10][1]

tensor([1., 1., 1., 1., 1., 1., 0., 1., 1., 1.])

In [83]:
y = model.head.target_labels.reshape(1, -1)
y = torch.repeat_interleave(y, 10, dim=0).reshape(-1, 1)

In [84]:
model(xval,y).reshape(-1,2).min(dim=1)

torch.return_types.min(
values=tensor([0.2119, 0.2482, 0.1485, 0.2477, 0.2495, 0.0429, 0.2111, 0.2406, 0.2477,
        0.0179], dtype=torch.float64, grad_fn=<MinBackward0>),
indices=tensor([1, 0, 1, 1, 0, 1, 0, 0, 0, 1]))

In [9]:
model.head.num_qubits

3

In [74]:
x = torch.zeros(10)

In [75]:
idx = list(range(10))

In [76]:
idx

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [77]:
x[0] = 1

In [78]:
x

tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [79]:
available_idx = list(range(10))

In [80]:
available_idx

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [81]:
available_idx.remove(0)
# available_idx.remove(9)

In [82]:
available_idx

[1, 2, 3, 4, 5, 6, 7, 8, 9]

In [83]:
used_index = [0]

In [92]:
torch.abs(torch.tensor(available_idx).reshape(-1,1)- torch.tensor(used_index)).prod(1)

tensor([1, 2, 3, 4, 5, 6, 7, 8, 9])

In [85]:
k.argmax().item()

8

In [ ]:
num_classes = 8
num_qubits = 3

def get_target_states() -> torch.Tensor:
    """Computes the target states associtated with the different labels

    Raises:
        ValueError: _description_

    Returns:
        torch.Tensor: target states
    """

    
    def get_ideal_state(used_index, available_index) -> int:
        us_idx = torch.tensor(used_index)
        av_idx = torch.tensor(available_index).reshape(-1, 1)
        prod_distance = torch.abs(av_idx - us_idx).prod(1)
        return prod_distance.argmax().item()


    target_states = torch.zeros(num_classes, 2**num_qubits)
    available_index = list(range(2**num_qubits))

    used_index = []
    for idx in range(num_classes):
        if idx == 0:
            state_idx = 0
        else:
            state_idx = available_index[get_ideal_state(used_index, available_index)]

        target_states[idx, state_idx] = 1
        used_index.append(state_idx)
        available_index.remove(state_idx)
    return target_states

get_target_states()

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0.]])

In [89]:
available_idx

[1, 2, 3, 4, 5, 6, 7, 8, 9]